# BraTS 2024 - U-Net Model Training (Local Workstation)

This notebook trains a 2D U-Net model on preprocessed BraTS MRI slices:
1. Loads the preprocessed dataset (`train` and `validate` splits).
2. Initializes 2D U-Net (4 input channels $\to$ 4 output classes).
3. Trains using combined `DiceCELoss` and `AdamW` optimizer.
4. Automatically saves the best model checkpoint based on validation Mean Dice score.
5. Plots loss and Dice metric curves.

## 1. Environment and repository setup

In [ ]:
import sys
from pathlib import Path

# Automatically resolve repository root
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
repo_src = repo_root / "src"

if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

import torch
import matplotlib.pyplot as plt

from brain_mri.data.dataset import create_dataloader
from brain_mri.models.unet import UNet
from brain_mri.models.losses import DiceCELoss
from brain_mri.training import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## 2. Initialize DataLoaders

In [ ]:
# Path to preprocessed 2D slices
PROCESSED_ROOT = repo_root / "data" / "processed"
train_dir = PROCESSED_ROOT / "train"
val_dir = PROCESSED_ROOT / "validate"

# Check dataset existence
assert train_dir.exists(), f"Train directory missing: {train_dir}. Please run 01_dataset_generation.ipynb first!"
assert val_dir.exists(), f"Validation directory missing: {val_dir}. Please run 01_dataset_generation.ipynb first!"

BATCH_SIZE = 16
NUM_WORKERS = 4  # Set to 2 or 4 depending on CPU cores

train_loader = create_dataloader(
    split_dir=train_dir,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = create_dataloader(
    split_dir=val_dir,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train samples:      {len(train_loader.dataset)} (batches: {len(train_loader)})")
print(f"Validation samples: {len(val_loader.dataset)} (batches: {len(val_loader)})")

## 3. Model, Loss, Optimizer and Trainer initialization

In [ ]:
# Directory to store best model weights
CHECKPOINT_DIR = repo_root / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# 4 MRI input channels (T1, T1CE, T2, FLAIR) -> 4 BraTS classes (Background, NCR, ED, ET)
model = UNet(in_channels=4, num_classes=4, base_features=64)

criterion = DiceCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
)

print("Trainer initialized successfully!")

## 4. Run Model Training

In [ ]:
EPOCHS = 30
history = trainer.fit(epochs=EPOCHS)

## 5. Learning curves visualization

In [ ]:
plt.figure(figsize=(14, 5))

# 1. Loss curves
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", color="tab:blue")
plt.plot(history["val_loss"], label="Validation Loss", color="tab:orange")
plt.xlabel("Epoch")
plt.ylabel("DiceCE Loss")
plt.title("Loss Curves")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)

# 2. Validation Mean Dice
plt.subplot(1, 2, 2)
plt.plot(history["val_mean_dice"], label="Val Mean Dice", color="tab:green", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Mean Dice (NCR, ED, ET)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
curves_path = CHECKPOINT_DIR / "learning_curves.png"
plt.savefig(curves_path, dpi=200)
print(f"Saved learning curves to: {curves_path}")
plt.show()